## 📦 1. Install Required Packages

In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install scikit-learn matplotlib seaborn tqdm pillow pandas

import IPython
IPython.Application.instance().kernel.do_shutdown(True)

## 📚 2. Import Libraries

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")

## ⚙️ 3. Configuration and Hyperparameters

In [ ]:
class Config:
    # Auto-detect the correct dataset path structure
    base_path = Path('/kaggle/input/citrus-blackspot-classification')
    
    # Check if dataset has nested folder (common when ZIP is extracted)
    if (base_path / 'citrus_blackspot_classification' / 'train').exists():
        DATA_DIR = base_path / 'citrus_blackspot_classification'
    elif (base_path / 'train').exists():
        DATA_DIR = base_path
    else:
        # Fallback: list what's actually there for debugging
        DATA_DIR = base_path
    
    OUTPUT_DIR = Path('/kaggle/working')
    
    MODEL_NAME = 'densenet121'
    NUM_CLASSES = 2
    IMAGE_SIZE = 224
    
    BATCH_SIZE = 32
    NUM_EPOCHS = 50
    LEARNING_RATE = 0.001
    WEIGHT_DECAY = 1e-4
    
    CLASS_WEIGHTS = None
    
    NUM_WORKERS = 0
    PIN_MEMORY = True
    
    EARLY_STOPPING_PATIENCE = 10
    REDUCE_LR_PATIENCE = 5
    
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

config = Config()
config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset Directory: {config.DATA_DIR.absolute()}")
print(f"Dataset Exists: {config.DATA_DIR.exists()}")

# Debug: Show actual directory structure
if config.DATA_DIR.exists():
    print(f"\n📂 Contents of {config.DATA_DIR}:")
    for item in sorted(config.DATA_DIR.iterdir()):
        print(f"  - {item.name}{'/' if item.is_dir() else ''}")
    
    # Check if train/val/test folders exist
    for split in ['train', 'val', 'test']:
        split_path = config.DATA_DIR / split
        if split_path.exists():
            print(f"\n📂 Contents of {split}:")
            for item in sorted(split_path.iterdir()):
                if item.is_dir():
                    count = len(list(item.glob('*')))
                    print(f"  - {item.name}/: {count} files")

print(f"\nOutput Directory: {config.OUTPUT_DIR.absolute()}")
print(f"Device: {config.DEVICE}")
print(f"Model: {config.MODEL_NAME}")
print(f"Batch Size: {config.BATCH_SIZE}")
print(f"Image Size: {config.IMAGE_SIZE}")
print(f"Learning Rate: {config.LEARNING_RATE}")
print(f"Number of Epochs: {config.NUM_EPOCHS}")


## 📁 4. Dataset Class Definition

In [ ]:
class CitrusBlackSpotDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        
        self.classes = ['blackspot', 'healthy']
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}
        
        self.images = []
        self.labels = []
        
        # Debug: Check if root_dir exists
        if not self.root_dir.exists():
            print(f"⚠️ WARNING: {self.root_dir} does not exist!")
            print(f"Available paths:")
            parent = self.root_dir.parent
            if parent.exists():
                for item in sorted(parent.iterdir()):
                    print(f"  - {item}")
        
        for class_name in self.classes:
            class_dir = self.root_dir / class_name
            if class_dir.exists():
                image_count = 0
                for img_path in class_dir.glob('*'):
                    if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                        self.images.append(str(img_path))
                        self.labels.append(self.class_to_idx[class_name])
                        image_count += 1
                print(f"  ✅ Found {image_count} images in {class_name}/")
            else:
                print(f"  ❌ Directory not found: {class_dir}")
        
        print(f"\nTotal: Found {len(self.images)} images in {self.root_dir}")
        print(f"Class distribution: {Counter(self.labels)}")
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image, label


## 🔄 5. Data Augmentation and Transformations

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(config.IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.1
    ),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.1, 0.1),
        scale=(0.9, 1.1)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((config.IMAGE_SIZE, config.IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("✅ Transformations defined")

## 📊 6. Create Data Loaders

In [ ]:
train_dataset = CitrusBlackSpotDataset(
    config.DATA_DIR / 'train',
    transform=train_transform
)

val_dataset = CitrusBlackSpotDataset(
    config.DATA_DIR / 'val',
    transform=val_transform
)

test_dataset = CitrusBlackSpotDataset(
    config.DATA_DIR / 'test',
    transform=val_transform
)

if len(train_dataset) == 0:
    raise ValueError(
        "❌ No training images found! Please check:\n"
        "1. Dataset structure should be: /kaggle/input/citrus-blackspot-classification/train/blackspot/ and /train/healthy/\n"
        "2. Images should be .jpg, .jpeg, or .png format\n"
        "3. Dataset is properly uploaded to Kaggle"
    )

train_labels = train_dataset.labels
class_counts = Counter(train_labels)
total_samples = len(train_labels)

class_weights = [total_samples / (len(class_counts) * class_counts[i]) for i in range(len(class_counts))]
config.CLASS_WEIGHTS = class_weights

print(f"\n📊 Class Distribution:")
for i, class_name in enumerate(train_dataset.classes):
    count = class_counts[i]
    percentage = (count / total_samples) * 100
    print(f"   {class_name}: {count} images ({percentage:.1f}%) - weight: {class_weights[i]:.4f}")

train_loader = DataLoader(
    train_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=config.NUM_WORKERS,
    pin_memory=config.PIN_MEMORY
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=config.PIN_MEMORY
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=config.PIN_MEMORY
)

print(f"\n✅ Data loaders created")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

## 🖼️ 7. Visualize Sample Images

In [ ]:
def show_batch(loader, n_images=8):
    images, labels = next(iter(loader))
    
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    images = images * std + mean
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    
    class_names = ['Black Spot', 'Healthy']
    
    for i in range(min(n_images, len(images))):
        img = images[i].permute(1, 2, 0).cpu().numpy()
        img = np.clip(img, 0, 1)
        
        label_name = class_names[labels[i]]
        color = 'red' if labels[i] == 0 else 'green'
        
        axes[i].imshow(img)
        axes[i].set_title(label_name, color=color, fontweight='bold', fontsize=12)
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.savefig(config.OUTPUT_DIR / 'sample_images.png', dpi=150, bbox_inches='tight')
    plt.show()

show_batch(train_loader)

## 🏗️ 8. Define Model Architecture

In [ ]:
def create_model(model_name='densenet121', num_classes=2, pretrained=True):
    if model_name == 'resnet50':
        model = models.resnet50(pretrained=pretrained)
        num_features = model.fc.in_features
        model.fc = nn.Linear(num_features, num_classes)
    
    elif model_name == 'resnet34':
        model = models.resnet34(pretrained=pretrained)
        num_features = model.fc.in_features
        model.fc = nn.Linear(num_features, num_classes)
    
    elif model_name == 'densenet121':
        model = models.densenet121(pretrained=pretrained)
        num_features = model.classifier.in_features
        model.classifier = nn.Linear(num_features, num_classes)
    
    elif model_name == 'efficientnet_b0':
        model = models.efficientnet_b0(pretrained=pretrained)
        num_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(num_features, num_classes)
    
    elif model_name == 'mobilenet_v2':
        model = models.mobilenet_v2(pretrained=pretrained)
        num_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(num_features, num_classes)
    
    else:
        raise ValueError(f"Unknown model: {model_name}")
    
    return model

model = create_model(
    model_name=config.MODEL_NAME,
    num_classes=config.NUM_CLASSES,
    pretrained=True
)

model = model.to(config.DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n✅ Model: {config.MODEL_NAME}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 🎯 9. Define Loss Function and Optimizer

In [ ]:
class_weights_tensor = torch.tensor(config.CLASS_WEIGHTS, dtype=torch.float32).to(config.DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

print(f"✅ Using class weights: {config.CLASS_WEIGHTS}")
for i, class_name in enumerate(train_dataset.classes):
    print(f"   - {class_name}: weight = {config.CLASS_WEIGHTS[i]:.4f}")

optimizer = optim.Adam(
    model.parameters(),
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=config.REDUCE_LR_PATIENCE,
    verbose=True
)

print("\n✅ Loss function, optimizer, and scheduler defined")

## 🏋️ 10. Training and Validation Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc='Training')
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{100.*correct/total:.2f}%'
        })
    
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc


def validate_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        pbar = tqdm(loader, desc='Validation')
        for images, labels in pbar:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100.*correct/total:.2f}%'
            })
    
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc, all_preds, all_labels

print("✅ Training and validation functions defined")

## 🚀 11. Train the Model

In [ ]:
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

best_val_acc = 0.0
best_model_path = config.OUTPUT_DIR / 'best_model.pth'
patience_counter = 0

print("\n" + "="*70)
print("🚀 STARTING TRAINING")
print("="*70)

for epoch in range(config.NUM_EPOCHS):
    print(f"\n📅 Epoch {epoch+1}/{config.NUM_EPOCHS}")
    print("-" * 70)
    
    train_loss, train_acc = train_epoch(
        model, train_loader, criterion, optimizer, config.DEVICE
    )
    
    val_loss, val_acc, val_preds, val_labels = validate_epoch(
        model, val_loader, criterion, config.DEVICE
    )
    
    scheduler.step(val_loss)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"\n📊 Epoch {epoch+1} Summary:")
    print(f"   Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"   Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
    print(f"   LR: {optimizer.param_groups[0]['lr']:.6f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'val_loss': val_loss,
        }, best_model_path)
        print(f"   ✅ New best model saved! (Val Acc: {val_acc:.2f}%)")
        patience_counter = 0
    else:
        patience_counter += 1
    
    if patience_counter >= config.EARLY_STOPPING_PATIENCE:
        print(f"\n⏹️ Early stopping triggered after {epoch+1} epochs")
        break
    
    if (epoch + 1) % 10 == 0:
        checkpoint_path = config.OUTPUT_DIR / f'checkpoint_epoch_{epoch+1}.pth'
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, checkpoint_path)

print("\n" + "="*70)
print("✨ TRAINING COMPLETE!")
print(f"🏆 Best Validation Accuracy: {best_val_acc:.2f}%")
print("="*70)

## 📈 12. Plot Training History

In [ ]:
def plot_training_history(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    ax1.plot(epochs, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
    ax1.plot(epochs, history['val_loss'], 'r-', label='Val Loss', linewidth=2)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(epochs, history['train_acc'], 'b-', label='Train Acc', linewidth=2)
    ax2.plot(epochs, history['val_acc'], 'r-', label='Val Acc', linewidth=2)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    ax2.set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(config.OUTPUT_DIR / 'training_history.png', dpi=300, bbox_inches='tight')
    plt.show()

plot_training_history(history)

## 🧪 13. Evaluate on Test Set

In [ ]:
checkpoint = torch.load(best_model_path)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✅ Loaded best model from epoch {checkpoint['epoch']+1}")
print(f"   Best Val Acc: {checkpoint['val_acc']:.2f}%")

print("\n" + "="*70)
print("🧪 EVALUATING ON TEST SET")
print("="*70 + "\n")

test_loss, test_acc, test_preds, test_labels = validate_epoch(
    model, test_loader, criterion, config.DEVICE
)

print(f"\n📊 Test Results:")
print(f"   Test Loss: {test_loss:.4f}")
print(f"   Test Accuracy: {test_acc:.2f}%")

## 📊 14. Classification Report and Metrics

In [ ]:
class_names = ['Black Spot', 'Healthy']
print("\n" + "="*70)
print("📊 CLASSIFICATION REPORT")
print("="*70 + "\n")
print(classification_report(test_labels, test_preds, target_names=class_names))

precision, recall, f1, support = precision_recall_fscore_support(
    test_labels, test_preds, average='weighted'
)

print("\n" + "="*70)
print("📈 OVERALL METRICS")
print("="*70)
print(f"Accuracy:  {test_acc:.2f}%")
print(f"Precision: {precision*100:.2f}%")
print(f"Recall:    {recall*100:.2f}%")
print(f"F1-Score:  {f1*100:.2f}%")
print("="*70)

print("\n" + "="*70)
print("📊 CLASS-SPECIFIC METRICS")
print("="*70)
for i, class_name in enumerate(class_names):
    class_mask = np.array(test_labels) == i
    class_preds = np.array(test_preds)[class_mask]
    class_labels = np.array(test_labels)[class_mask]
    class_acc = (class_preds == class_labels).sum() / len(class_labels) * 100
    print(f"{class_name}: {class_acc:.2f}% accuracy ({len(class_labels)} samples)")
print("="*70)

## 🎨 15. Confusion Matrix Visualization

In [ ]:
def plot_confusion_matrix(y_true, y_pred, class_names):
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names,
        cbar_kws={'label': 'Count'},
        annot_kws={'size': 14, 'weight': 'bold'}
    )
    
    plt.ylabel('True Label', fontsize=12, fontweight='bold')
    plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
    plt.title('Confusion Matrix - Test Set', fontsize=14, fontweight='bold', pad=20)
    
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            percentage = cm[i, j] / cm[i].sum() * 100
            plt.text(
                j + 0.5, i + 0.7,
                f'({percentage:.1f}%)',
                ha='center', va='center',
                fontsize=10, color='gray'
            )
    
    plt.tight_layout()
    plt.savefig(config.OUTPUT_DIR / 'confusion_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()

plot_confusion_matrix(test_labels, test_preds, class_names)

## 💾 16. Save Final Model and Results

In [ ]:
final_model_path = config.OUTPUT_DIR / 'citrus_blackspot_detection_model.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'model_name': config.MODEL_NAME,
    'num_classes': config.NUM_CLASSES,
    'image_size': config.IMAGE_SIZE,
    'class_names': class_names,
    'class_weights': config.CLASS_WEIGHTS,
    'test_accuracy': test_acc,
    'best_val_accuracy': best_val_acc,
}, final_model_path)

print(f"\n✅ Final model saved to: {final_model_path}")

history_df = pd.DataFrame(history)
history_df.to_csv(config.OUTPUT_DIR / 'training_history.csv', index=False)
print(f"✅ Training history saved")

results = {
    'test_accuracy': test_acc,
    'test_loss': test_loss,
    'best_val_accuracy': best_val_acc,
    'precision': precision * 100,
    'recall': recall * 100,
    'f1_score': f1 * 100,
}

results_df = pd.DataFrame([results])
results_df.to_csv(config.OUTPUT_DIR / 'test_results.csv', index=False)
print(f"✅ Test results saved")

print("\n" + "="*70)
print("🎉 ALL DONE!")
print("="*70)
print(f"\n📁 All outputs saved to: {config.OUTPUT_DIR}")
print("\nGenerated files:")
print("  📊 training_history.png")
print("  🎨 confusion_matrix.png")
print("  🖼️  sample_images.png")
print("  💾 best_model.pth")
print("  💾 citrus_blackspot_detection_model.pth")
print("  📄 training_history.csv")
print("  📄 test_results.csv")
print("\n" + "="*70)

## 🔮 17. Inference Function

In [ ]:
def predict_image(model, image_path, transform, device, class_names):
    model.eval()
    
    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        outputs = model(image_tensor)
        probabilities = torch.nn.functional.softmax(outputs, dim=1)
        confidence, predicted = torch.max(probabilities, 1)
    
    pred_class = class_names[predicted.item()]
    confidence = confidence.item() * 100
    
    plt.figure(figsize=(10, 6))
    plt.imshow(image)
    plt.axis('off')
    
    color = 'green' if pred_class == 'Healthy' else 'red'
    plt.title(
        f"Prediction: {pred_class}\nConfidence: {confidence:.2f}%",
        fontsize=14,
        fontweight='bold',
        color=color,
        pad=20
    )
    plt.tight_layout()
    plt.show()
    
    return pred_class, confidence

print("✅ Inference function defined")

## 📝 18. Summary and Next Steps

### 🎉 Training Complete!

**Model Performance:**
- Test Accuracy: Check the output above
- Model Architecture: DenseNet-121 with transfer learning
- Total Parameters: ~7M
- Class Weighting: Applied for balanced learning

### 📦 Deliverables

1. **Trained Model:** `citrus_blackspot_detection_model.pth`
2. **Best Checkpoint:** `best_model.pth`
3. **Training Curves:** `training_history.png`
4. **Confusion Matrix:** `confusion_matrix.png`
5. **Metrics CSV:** `test_results.csv`

### 🚀 Next Steps

1. Download the model from Kaggle output directory
2. Integrate into API (FRESH ML backend)
3. Deploy to production server
4. Test with real images from field
5. Monitor performance and collect feedback

### 🔧 Model Optimization

If you need better performance:
- Try different architectures (ResNet50, EfficientNet, Vision Transformer)
- Increase training epochs
- Use more data augmentation
- Experiment with different class weights
- Use ensemble methods
- Collect more training data

### 📊 Dataset Notes

**Dataset:** 1,461 citrus fruit images
- Train: 1,022 images (70%)
- Validation: 219 images (15%)
- Test: 220 images (15%)

**Strategies Applied:**
- Automatic class weighting based on inverse frequency
- Extensive data augmentation
- Early stopping to prevent overfitting
- Learning rate reduction on plateau

---

**Created for:** FRESH ML - Module 2 (Disease Detection)  
**Date:** November 2025  
**Model:** Citrus Black Spot Detection  
**Task:** Binary Classification (Healthy vs Black Spot)